# Practical 2: Implement the A* Search Algorithm to Solve the 8-Puzzle Problem

## Aim
Implement A* search with an appropriate heuristic to find a solution to the 8-Puzzle problem.

In [1]:
# Cell 1: 8-Puzzle Problem Representation and A* Search Implementation
import heapq

class PuzzleNode:
    """Node representation in the A* state-space search graph."""
    def __init__(self, state, g=0, h=0, parent=None, move=""):
        self.state = state  # Tuple of 9 integers representing the 3x3 board
        self.g = g          # Path cost g(n)
        self.h = h          # Heuristic estimate h(n)
        self.f = g + h      # Evaluation function f(n) = g(n) + h(n)
        self.parent = parent
        self.move = move

    def __lt__(self, other):
        # Tie-breaking prefers lower f, then higher g (closer to goal)
        if self.f == other.f:
            return self.g > other.g
        return self.f < other.f

# Standard 8-Puzzle Goal State
GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8, 0)
GOAL_COORDS = {
    1: (0, 0), 2: (0, 1), 3: (0, 2),
    4: (1, 0), 5: (1, 1), 6: (1, 2),
    7: (2, 0), 8: (2, 1), 0: (2, 2)
}

def manhattan_distance(state):
    """Calculates the Manhattan distance heuristic for non-blank tiles."""
    total_dist = 0
    for idx, tile in enumerate(state):
        if tile != 0:
            curr_r, curr_c = idx // 3, idx % 3
            goal_r, goal_c = GOAL_COORDS[tile]
            total_dist += abs(curr_r - goal_r) + abs(curr_c - goal_c)
    return total_dist

def is_solvable(state):
    """Determines solvability using inversion count (solvable if even)."""
    tiles = [val for val in state if val != 0]
    inversions = 0
    for i in range(len(tiles)):
        for j in range(i + 1, len(tiles)):
            if tiles[i] > tiles[j]:
                inversions += 1
    return (inversions % 2) == 0

def get_legal_neighbors(node):
    """Generates legal board transitions by moving the blank tile (0)."""
    neighbors = []
    blank_idx = node.state.index(0)
    r, c = blank_idx // 3, blank_idx % 3
    moves = [
        ('Up', -1, 0),
        ('Down', 1, 0),
        ('Left', 0, -1),
        ('Right', 0, 1)
    ]
    for move_name, dr, dc in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_blank_idx = nr * 3 + nc
            new_state = list(node.state)
            # Swap blank tile with adjacent tile
            new_state[blank_idx], new_state[new_blank_idx] = new_state[new_blank_idx], new_state[blank_idx]
            new_state_tuple = tuple(new_state)
            g_cost = node.g + 1
            h_cost = manhattan_distance(new_state_tuple)
            neighbors.append(PuzzleNode(
                new_state_tuple,
                g=g_cost,
                h=h_cost,
                parent=node,
                move=f"{move_name} (swap with tile {node.state[new_blank_idx]})"
            ))
    return neighbors

def a_star_search(initial_state):
    """Executes A* search with priority queue and Manhattan heuristic."""
    if not is_solvable(initial_state):
        return None, 0, 0, False

    root = PuzzleNode(initial_state, g=0, h=manhattan_distance(initial_state))
    open_heap = [root]
    best_g = {initial_state: 0}
    expanded_count = 0
    generated_count = 1

    while open_heap:
        current = heapq.heappop(open_heap)

        if current.state == GOAL_STATE:
            # Reconstruct optimal path from goal to root
            path = []
            curr = current
            while curr:
                path.append(curr)
                curr = curr.parent
            path.reverse()
            return path, expanded_count, generated_count, True

        if current.g > best_g.get(current.state, float('inf')):
            continue

        expanded_count += 1

        for neighbor in get_legal_neighbors(current):
            generated_count += 1
            if neighbor.g < best_g.get(neighbor.state, float('inf')):
                best_g[neighbor.state] = neighbor.g
                heapq.heappush(open_heap, neighbor)

    return None, expanded_count, generated_count, False


In [2]:
# Cell 2: Execution, State-by-State Visualization, and Verification

def format_board(state):
    rows = []
    for r in range(3):
        row_vals = [' ' if val == 0 else str(val) for val in state[r*3:(r+1)*3]]
        rows.append("  " + " | ".join(row_vals))
    return "\n  ---+---+---\n".join(rows)

# Test Case 1: Solvable configuration
initial_board = (1, 2, 3, 0, 4, 6, 7, 5, 8)
print("=" * 65)
print("TEST CASE 1: SOLVABLE 8-PUZZLE EXECUTION")
print("=" * 65)
print("Initial Board State:")
print(format_board(initial_board))
print(f"Initial Manhattan Distance: {manhattan_distance(initial_board)}")
print("-" * 65)

solution_path, expanded_nodes, generated_nodes, is_success = a_star_search(initial_board)

if is_success:
    print(f"Optimal Solution Found in {len(solution_path) - 1} moves!\n")
    for step_idx, node in enumerate(solution_path):
        if step_idx == 0:
            print(f"Step 0 (Initial State):")
        else:
            print(f"Step {step_idx} (Action: {node.move}) [g={node.g}, h={node.h}, f={node.f}]:")
        print(format_board(node.state))
        print()

    print("-" * 65)
    print(f"Total Moves Required : {len(solution_path) - 1}")
    print(f"Expanded States      : {expanded_nodes}")
    print(f"Generated States     : {generated_nodes}")
    print(f"Goal Verification    : {solution_path[-1].state == GOAL_STATE} (Matches Goal Configuration Exactly)")
    assert solution_path[-1].state == GOAL_STATE, "Verification Failed: Final board does not match goal!"

# Test Case 2: Unsolvable configuration test
unsolvable_board = (1, 2, 3, 4, 5, 6, 8, 7, 0)
print("\n" + "=" * 65)
print("TEST CASE 2: UNSOLVABLE 8-PUZZLE VALIDATION")
print("=" * 65)
print("Board Configuration:")
print(format_board(unsolvable_board))
solvability = is_solvable(unsolvable_board)
print(f"Inversion Solvability Check: {solvability} (Odd inversions detect unsolvability)")
path_unsolvable, _, _, success_unsolvable = a_star_search(unsolvable_board)
print(f"Solver Handled Gracefully  : Solvable = {success_unsolvable}")
assert not success_unsolvable, "Verification Failed: Unsolvable board should not succeed!"
print("Verification Status        : ALL CHECKS PASSED")


TEST CASE 1: SOLVABLE 8-PUZZLE EXECUTION
Initial Board State:
  1 | 2 | 3
  ---+---+---
    | 4 | 6
  ---+---+---
  7 | 5 | 8
Initial Manhattan Distance: 3
-----------------------------------------------------------------
Optimal Solution Found in 3 moves!

Step 0 (Initial State):
  1 | 2 | 3
  ---+---+---
    | 4 | 6
  ---+---+---
  7 | 5 | 8

Step 1 (Action: Right (swap with tile 4)) [g=1, h=2, f=3]:
  1 | 2 | 3
  ---+---+---
  4 |   | 6
  ---+---+---
  7 | 5 | 8

Step 2 (Action: Down (swap with tile 5)) [g=2, h=1, f=3]:
  1 | 2 | 3
  ---+---+---
  4 | 5 | 6
  ---+---+---
  7 |   | 8

Step 3 (Action: Right (swap with tile 8)) [g=3, h=0, f=3]:
  1 | 2 | 3
  ---+---+---
  4 | 5 | 6
  ---+---+---
  7 | 8 |  

-----------------------------------------------------------------
Total Moves Required : 3
Expanded States      : 3
Generated States     : 11
Goal Verification    : True (Matches Goal Configuration Exactly)

TEST CASE 2: UNSOLVABLE 8-PUZZLE VALIDATION
Board Configuration:
  1 | 2 |